# S1-S5 OLS 점검

이 노트북은 스마트팜 센서 데이터에서 **S1(전체 구간 레벨 OLS)**, **S2(펌프 작동 구간 + HAC + 1차 차분)**, **S3(VIF 정리용 축소 모델 비교)**, **S5(lag feature 스캔)** 을 순서대로 점검하기 위한 작업 노트입니다.

핵심 원칙은 다음과 같습니다.

1. `S1`은 빠른 스크리닝용 베이스라인입니다. 높은 `R2`가 나와도 그대로 믿지 않습니다.
2. `S2`는 펌프가 실제로 도는 구간만 남기고, 자기상관에 덜 취약한 HAC 표준오차와 1차 차분을 함께 점검합니다.
3. `DW`가 매우 낮거나 `VIF`가 매우 크면, 그 모델은 좋은 설명모형이라기보다 추세나 상태전이를 따라간 결과일 수 있습니다.
4. 비율형 타깃(`zone1_resistance`, `wire_to_water_efficiency`)은 정의식의 영향이 남기 때문에 더 보수적으로 해석합니다.
5. `S3`는 `DIFF_on`에서도 공선성이 큰 타깃만 다시 다루며, 변수 제거가 정말 정당한지는 `AIC/BIC` 손실까지 같이 봅니다.
6. `S5`는 lag를 넣어 잔차 자기상관을 줄일 수 있는지 보는 단계입니다. 단, `DW`가 좋아져도 `VIF`가 급등하면 계수 해석은 여전히 조심해야 합니다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 입력 CSV 후보 경로입니다. 앞에서부터 존재하는 경로를 사용합니다.
INPUT_PATHS = [
    Path("human_A/src/selected_smartfarm.csv"),
    Path("src/selected_smartfarm.csv"),
    Path("selected_smartfarm.csv"),
]

# 결과 CSV 파일명입니다. 실제 저장 위치는 실행 폴더에 맞춰 아래 helper에서 결정합니다.
OUTPUT_S1_NAME = "ols_first_pass.csv"
OUTPUT_S2_NAME = "ols_first_pass_v2.csv"
OUTPUT_S3_NAME = "ols_s3_vif_cleanup.csv"
OUTPUT_S5_RESULTS_NAME = "ols_s5_lag_results.csv"
OUTPUT_S5_HISTORY_NAME = "ols_s5_lag_history.csv"

# 타깃별 설명변수 사전입니다.
# 아래 주석에는 왜 넣었는지, 왜 뺐는지를 함께 기록해 둡니다.
TARGET_DICTIONARY = {
    # ============================================================
    # 1) motor_current_a
    # ============================================================
    "motor_current_a": [
        # "motor_power_kw",         # ❌ 제외: P=V·I 물리 항등식이라 전류와 직접 묶입니다.
        #                           #    회귀라기보다 공식을 다시 확인하는 셈이어서 제외합니다.
        # "wire_to_water_efficiency", # ❌ 제외: 파생변수이며 motor_power_kw를 분모로 포함합니다.
        #                           #    target과 간접 순환 구조가 생겨 해석이 흐려집니다.
        "motor_temperature_c",      # ✅ 유지: 권선 저항과 발열을 반영하는 비교적 독립적인 원인 후보입니다.
        "pump_rpm",                 # ✅ 유지: 외생적 운전점 제어 변수로 해석 가능합니다.
        "discharge_pressure_kpa",   # ✅ 유지: 펌프 부하를 대표하는 물리량입니다.
    ],

    # ============================================================
    # 2) zone1_resistance (= zone1_pressure / zone1_flow)
    # ============================================================
    "zone1_resistance": [
        "zone1_pressure_kpa",       # ⚠️ 유지: 정의식의 분자입니다.
        #                           #    이 모델은 순수 예측이 아니라 분자 기여도를 보는 용도로 해석합니다.
        "zone1_flow_l_min",         # ⚠️ 유지: 정의식의 분모입니다.
        #                           #    따라서 계수 해석은 '저항 예측'보다 '분자·분모 분해'에 가깝습니다.
    ],

    # ============================================================
    # 3) wire_to_water_efficiency (= hydraulic_power / motor_power)
    # ============================================================
    "wire_to_water_efficiency": [
        # "motor_power_kw",         # ❌ 제외: 효율 정의식의 분모라 직접적인 공선성 원인입니다.
        # "hydraulic_power_kw",     # ❌ 제외: 효율 정의식의 분자라 직접적인 공선성 원인입니다.
        # "differential_pressure_kpa", # ❌ 제외: hydraulic_power와 매우 강하게 얽힐 가능성이 큽니다.
        # "flow_rate_l_min",        # ❌ 제외: hydraulic_power 구성 성분이어서 정의식 오염이 남습니다.
        "pump_rpm",                 # ✅ 유지: 운전점 변화 자체를 반영하는 외생 변수입니다.
        "motor_temperature_c",      # ✅ 유지: 전기적/열적 손실을 반영합니다.
        "suction_pressure_kpa",     # ✅ 유지: 흡입 상태와 캐비테이션 징후를 간접 반영합니다.
        "bearing_temperature_c",    # ✅ 유지: 기계적 손실과 마찰 증가를 반영합니다.
    ],

    # ============================================================
    # 4) mix_ph
    # ============================================================
    "mix_ph": [
        # "mix_target_ph",          # ❌ 제외: 설정값이 사실상 상수라 설명변수 역할을 못합니다.
        "dosing_acid_ml_min",       # ✅ 유지: pH를 직접 바꾸는 주입량입니다.
        "mix_temp_c",               # ✅ 유지: pH 센서와 용액 반응의 온도 의존성을 반영합니다.
        "acid_tank_level_pct",      # ✅ 유지: 산액 부족 시 제어 실패 가능성을 반영합니다.
        "mix_flow_l_min",           # ✅ 유지: 희석 및 혼합량 변화를 반영합니다.
    ],

    # ============================================================
    # 5) mix_ec_ds_m
    # ============================================================
    "mix_ec_ds_m": [
        "mix_target_ec_ds_m",       # ✅ 유지: 이 변수는 실제로 값이 변하며 설정 농도를 반영합니다.
        "tank_a_level_pct",         # ✅ 유지: A 비료 잔량 변화가 공급 농도에 영향을 줄 수 있습니다.
        "tank_b_level_pct",         # ✅ 유지: B 비료 잔량 변화도 같은 이유로 포함합니다.
        "mix_temp_c",               # ✅ 유지: 전도도는 온도 영향이 비교적 큽니다.
        "mix_flow_l_min",           # ✅ 유지: 희석 정도를 반영합니다.
        "raw_water_temp_c",         # ✅ 유지: 원수 온도에 따른 배경 변화를 반영합니다.
    ],
}

# 비율형 타깃은 정의식의 영향을 직접 받으므로 해석 메모를 따로 둡니다.
TARGET_NOTES = {
    "zone1_resistance": "분자·분모 기여도 분해 관점으로만 해석해야 합니다.",
    "wire_to_water_efficiency": "운전점과 정의식 영향이 함께 남는 비율형 지표입니다.",
}


In [2]:
def resolve_existing_path(paths):
    """경로 후보 중 실제로 존재하는 첫 파일 경로를 반환합니다."""
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(paths)


def get_analysis_home():
    """노트북이 어느 폴더에서 실행되든 human_A 작업 폴더를 찾아 반환합니다."""
    cwd = Path.cwd()
    if (cwd / "src").exists() and (cwd / "H2_preregistered_spec.md").exists():
        return cwd
    if (cwd / "human_A").exists() and (cwd / "human_A" / "src").exists():
        return cwd / "human_A"
    return cwd


def resolve_output_path(filename):
    """실행 위치와 무관하게 결과 CSV를 human_A 폴더 기준으로 저장할 경로를 반환합니다."""
    return get_analysis_home() / filename


def resolve_existing_output_path(filename):
    """이미 저장된 결과 CSV를 실행 위치와 무관하게 찾아 반환합니다."""
    analysis_home = get_analysis_home()
    candidates = [
        analysis_home / filename,
        Path(filename),
        Path("human_A") / filename,
    ]
    return resolve_existing_path(candidates)


def load_source_frame(path):
    """원본 CSV를 불러와 timestamp 기준으로 정렬합니다."""
    df = pd.read_csv(path, parse_dates=["timestamp"])
    return df.sort_values("timestamp")


def add_derived_columns(df):
    """분석에 필요한 파생변수를 생성합니다."""
    out = df.copy()
    out["hydraulic_power_kw"] = out["discharge_pressure_kpa"] * out["flow_rate_l_min"] / 60000
    out["differential_pressure_kpa"] = out["discharge_pressure_kpa"] - out["suction_pressure_kpa"]
    out["wire_to_water_efficiency"] = out["hydraulic_power_kw"] / out["motor_power_kw"].replace(0, np.nan)
    out["zone1_resistance"] = out["zone1_pressure_kpa"] / out["zone1_flow_l_min"].replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)


def compute_vif_table(X):
    """설명변수별 VIF를 계산합니다."""
    if X.shape[1] == 0:
        return {}
    if X.shape[1] == 1:
        return {X.columns[0]: 1.0}

    values = X.astype(float).values
    return {
        column: float(variance_inflation_factor(values, idx))
        for idx, column in enumerate(X.columns)
    }


def build_quality_flags(r2, dw, max_vif):
    """적합 결과를 빠르게 읽기 위한 경고 문구를 만듭니다."""
    flags = []
    if dw < 1.0:
        flags.append("잔차 자기상관 심함")
    elif dw < 1.5:
        flags.append("잔차 자기상관 주의")

    if max_vif > 30:
        flags.append("다중공선성 매우 큼")
    elif max_vif > 10:
        flags.append("다중공선성 큼")

    if r2 > 0.95 and dw < 0.5:
        flags.append("추세 또는 상태전이 주도 적합 의심")

    return " | ".join(flags) if flags else "특이 경고 없음"


def build_target_note(target):
    """타깃별 해석 메모를 반환합니다."""
    return TARGET_NOTES.get(target, "물리적 연결과 운전 정책 영향을 함께 확인해야 합니다.")


def run_ols_s1(df, y_col, x_cols, tag="S1_LEVEL_all"):
    """S1용 전체 구간 레벨 OLS를 실행합니다."""
    d = df[[y_col] + x_cols].dropna()
    if d.empty:
        raise ValueError(f"dropna 이후 남은 행이 없습니다: {y_col}")

    X = sm.add_constant(d[x_cols], has_constant="add")
    res = sm.OLS(d[y_col], X).fit()
    dw = sm.stats.stattools.durbin_watson(res.resid)
    jb_stat, jb_p, _, _ = sm.stats.stattools.jarque_bera(res.resid)
    vif = compute_vif_table(d[x_cols])
    max_vif = float(max(vif.values())) if vif else np.nan

    return {
        "target": y_col,
        "variant": tag,
        "sample_scope": "전체 시점",
        "cov_type": "기본 OLS",
        "N": int(res.nobs),
        "R2": float(res.rsquared),
        "AdjR2": float(res.rsquared_adj),
        "F": float(res.fvalue),
        "F_p": float(res.f_pvalue),
        "AIC": float(res.aic),
        "BIC": float(res.bic),
        "DW": float(dw),
        "JB_p": float(jb_p),
        "JB_stat": float(jb_stat),
        "max_vif": max_vif,
        "quality_flags": build_quality_flags(float(res.rsquared), float(dw), max_vif),
        "target_note": build_target_note(y_col),
        "predictors": x_cols,
        "coef": {key: float(value) for key, value in res.params.drop("const", errors="ignore").items()},
        "pvals": {key: float(value) for key, value in res.pvalues.drop("const", errors="ignore").items()},
        "vif": vif,
    }


def diagnose_pump_on_rules(df, rpm_thresh=100, flow_thresh=1.0):
    """펌프 작동 구간 정의를 점검하고 기본 마스크를 반환합니다."""
    if "pump_rpm" in df.columns:
        rpm_mask = df["pump_rpm"] > rpm_thresh
        selected_mask = rpm_mask
        selected_rule = f"pump_rpm > {rpm_thresh}"
    else:
        rpm_mask = pd.Series(False, index=df.index)
        selected_mask = df["flow_rate_l_min"] > flow_thresh
        selected_rule = f"flow_rate_l_min > {flow_thresh}"

    flow_mask = df["flow_rate_l_min"] > flow_thresh
    mismatch_mask = rpm_mask != flow_mask if "pump_rpm" in df.columns else pd.Series(False, index=df.index)

    diagnostics = {
        "선택한 기준": selected_rule,
        "전체 행 수": int(len(df)),
        "pump_rpm 기준 on 행 수": int(rpm_mask.sum()) if "pump_rpm" in df.columns else np.nan,
        "flow_rate 기준 on 행 수": int(flow_mask.sum()),
        "불일치 행 수": int(mismatch_mask.sum()) if "pump_rpm" in df.columns else 0,
    }
    mismatch_preview = df.loc[mismatch_mask, ["timestamp", "pump_rpm", "flow_rate_l_min"]].head(10) if mismatch_mask.any() else pd.DataFrame()
    return selected_mask, diagnostics, mismatch_preview


def run_ols_s2(df, y_col, x_cols, tag, hac_lags=20):
    """S2용 HAC 표준오차 OLS를 실행합니다."""
    d = df[[y_col] + x_cols].dropna()
    if len(d) < 100:
        print(f"[{tag}] 건너뜀: N={len(d)} 부족")
        return None

    X = sm.add_constant(d[x_cols], has_constant="add")
    res = sm.OLS(d[y_col], X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    dw = sm.stats.stattools.durbin_watson(res.resid)
    jb_stat, jb_p, _, _ = sm.stats.stattools.jarque_bera(res.resid)
    vif = compute_vif_table(d[x_cols])
    max_vif = float(max(vif.values())) if vif else np.nan

    return {
        "target": y_col,
        "variant": tag,
        "sample_scope": "펌프 작동 구간" if tag == "LEVEL_on" else "펌프 작동 구간 1차 차분",
        "cov_type": f"HAC(maxlags={hac_lags})",
        "N": int(res.nobs),
        "R2": float(res.rsquared),
        "AdjR2": float(res.rsquared_adj),
        "AIC": float(res.aic),
        "BIC": float(res.bic),
        "DW": float(dw),
        "JB_p": float(jb_p),
        "JB_stat": float(jb_stat),
        "max_vif": max_vif,
        "quality_flags": build_quality_flags(float(res.rsquared), float(dw), max_vif),
        "target_note": build_target_note(y_col),
        "coef": {key: float(value) for key, value in res.params.drop("const", errors="ignore").items()},
        "pvals": {key: float(value) for key, value in res.pvalues.drop("const", errors="ignore").items()},
        "vif": vif,
    }


def recommend_variant(level_row, diff_row):
    """S2 두 변형 중 어느 쪽을 우선 볼지 추천하되, 남은 문제도 함께 표시합니다."""
    if level_row is None or diff_row is None:
        return "비교 불가", "필수 결과가 비어 있습니다."

    level_score = abs(level_row["DW"] - 2.0) + max(level_row["max_vif"] - 10.0, 0.0) / 10.0
    diff_score = abs(diff_row["DW"] - 2.0) + max(diff_row["max_vif"] - 10.0, 0.0) / 10.0

    if diff_score <= level_score:
        candidate_name = "DIFF_on"
        candidate_row = diff_row
        base_reason = "차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다."
    else:
        candidate_name = "LEVEL_on"
        candidate_row = level_row
        base_reason = "레벨 모델이 상대적으로 더 안정적입니다."

    if candidate_row["DW"] < 0.8 or candidate_row["max_vif"] > 30:
        return "추가 재설계 필요", base_reason + " 다만 자기상관 또는 공선성 문제가 여전히 커서 바로 해석하기 어렵습니다."

    if candidate_row["DW"] < 1.5 or candidate_row["max_vif"] > 10:
        return f"{candidate_name} (주의)", base_reason + " 다만 잔차 자기상관이나 공선성 문제가 일부 남아 있습니다."

    return candidate_name, base_reason + " 현재 진단상 가장 해석 가능한 후보입니다."


def judge_s3_candidate(base_row, candidate_row):
    """S3 축소 모델이 공선성 해소 대비 설명력 손실을 감수할 만한지 판정합니다."""
    delta_aic = float(candidate_row["AIC"] - base_row["AIC"])
    delta_bic = float(candidate_row["BIC"] - base_row["BIC"])
    candidate_max_vif = float(candidate_row["max_vif"])
    candidate_dw = float(candidate_row["DW"])

    if delta_aic <= -2 and candidate_max_vif < 4 and candidate_dw >= 1.5:
        verdict = "채택 후보"
        reason = "AIC가 개선되고 VIF와 DW도 함께 좋아졌습니다."
    elif delta_aic <= -2 and candidate_max_vif < 4:
        verdict = "조건부 후보"
        reason = "공선성은 해소됐지만 DW가 아직 낮아 추가 동적 모형 검토가 필요합니다."
    elif abs(delta_aic) < 2 and candidate_max_vif < 4:
        verdict = "단순화 가능"
        reason = "적합 손실이 크지 않으면서 공선성이 해소됐습니다."
    elif delta_aic >= 2 and candidate_max_vif < 4:
        verdict = "제거 비권장"
        reason = "VIF는 정리됐지만 AIC 손실이 커서 설명력 희생이 너무 큽니다."
    elif delta_aic >= 2:
        verdict = "제거 비권장"
        reason = "AIC도 악화되고 공선성 이득도 충분하지 않습니다."
    else:
        verdict = "추가 검토"
        reason = "일부 개선은 있지만 최종 채택 기준을 만족하지 못했습니다."

    return delta_aic, delta_bic, verdict, reason


# S2 이후 추가 실험 셀에서 그대로 재사용할 수 있도록 별칭을 맞춥니다.
run_ols = run_ols_s2


## S1. 전체 구간 베이스라인 OLS

S1은 **모든 시점**을 그대로 넣은 레벨 OLS입니다. 이 단계의 목적은 변수 후보를 빠르게 훑어보고, 어디에서 자기상관과 다중공선성이 심한지 확인하는 것입니다.

여기서 특히 조심해야 할 점은 다음과 같습니다.

- 펌프 정지 구간까지 포함되면 회귀가 물리 메커니즘보다 on/off 상태를 배우기 쉽습니다.
- 시계열 추세가 강하면 `R2`가 높아도 `DW`가 매우 낮을 수 있습니다.
- `max_vif > 10`이면 계수 해석이 불안정하므로, S1 결과는 진단용으로만 보는 편이 안전합니다.


In [3]:
# S1 실행: 전체 시점을 그대로 사용한 레벨 OLS입니다.
input_path_s1 = resolve_existing_path(INPUT_PATHS)
output_s1 = resolve_output_path(OUTPUT_S1_NAME)
df_base = load_source_frame(input_path_s1)
df_s1 = add_derived_columns(df_base)

s1_rows = []
for target, predictors in TARGET_DICTIONARY.items():
    s1_rows.append(run_ols_s1(df_s1, target, predictors))

s1_results = pd.DataFrame(s1_rows)
s1_results.to_csv(output_s1, index=False)

print(f"불러온 원본 데이터: {input_path_s1.resolve()}")
print(f"S1 저장 경로: {output_s1.resolve()}")
print(f"df_s1 shape: {df_s1.shape}")
display(s1_results[["target", "N", "R2", "AdjR2", "DW", "max_vif", "quality_flags"]])


불러온 원본 데이터: C:\Users\ui203\OneDrive\문서\green_project\human_A\src\selected_smartfarm.csv
S1 저장 경로: C:\Users\ui203\OneDrive\문서\green_project\human_A\ols_first_pass.csv
df_s1 shape: (129591, 49)


,target,N,R2,AdjR2,DW,max_vif,quality_flags
0,motor_current_a,129591,0.999683,0.999683,0.018046,75.157341,잔차 자기상관 심함 | 다중공선성 매우 큼 | 추세 또는 상태전이 주도 적합 의심
1,zone1_resistance,128720,0.185146,0.185133,0.339391,3.673862,잔차 자기상관 심함
2,wire_to_water_efficiency,129419,0.982434,0.982433,0.009025,9217.408279,잔차 자기상관 심함 | 다중공선성 매우 큼 | 추세 또는 상태전이 주도 적합 의심
3,mix_ph,129591,0.000017,-0.000014,0.035764,36.203439,잔차 자기상관 심함 | 다중공선성 매우 큼
4,mix_ec_ds_m,129591,0.814701,0.814693,0.032915,75716.404243,잔차 자기상관 심함 | 다중공선성 매우 큼


## S2. 펌프 작동 구간 + HAC + 1차 차분

S2는 S1보다 한 단계 더 보수적인 점검입니다.

이번에는 다음 판단을 반영합니다.

1. 펌프 작동 구간은 `H2_preregistered_spec.md`에 적어둔 H2 분석 설계 기준에 맞춰 **`pump_rpm > 100`** 을 기본 기준으로 사용합니다.
2. 참고로 `flow_rate_l_min > 1.0`과 거의 같지만, 시작 직후 몇 분은 `pump_rpm`은 켜져 있고 유량은 아직 1.0 미만일 수 있습니다. 이 불일치도 같이 기록합니다.
3. `LEVEL_on`은 펌프 작동 구간만 남긴 레벨 회귀이고, `DIFF_on`은 같은 구간을 1차 차분한 회귀입니다.
4. `DIFF_on`에서 `DW`가 2에 가까워지고 `VIF`가 줄어들면, 그쪽이 추세에 덜 끌린 결과라고 볼 수 있습니다.


In [4]:
# S2 준비: 원본, 펌프 작동 구간, 차분 프레임을 만듭니다.
input_path_s2 = resolve_existing_path(INPUT_PATHS)
df_raw = load_source_frame(input_path_s2)
on_mask, pump_rule_diag, mismatch_preview = diagnose_pump_on_rules(df_raw)
df_on = add_derived_columns(df_raw.loc[on_mask].copy()).set_index("timestamp")
df_diff = df_on.diff().dropna()

print("S2 펌프 작동 구간 진단")
display(pd.DataFrame([pump_rule_diag]))
if not mismatch_preview.empty:
    print("pump_rpm 기준과 flow_rate 기준이 다른 시작 구간 미리보기")
    display(mismatch_preview)

print(f"df_raw  : {df_raw.shape}")   # 기대: (129591, 45)
print(f"df_on   : {df_on.shape}")    # 기대: (65610, 48)
print(f"df_diff : {df_diff.shape}")  # 기대: (65609, 48)


S2 펌프 작동 구간 진단


,선택한 기준,전체 행 수,pump_rpm 기준 on 행 수,flow_rate 기준 on 행 수,불일치 행 수
0,pump_rpm > 100,129591,65610,65594,16


pump_rpm 기준과 flow_rate 기준이 다른 시작 구간 미리보기


,timestamp,pump_rpm,flow_rate_l_min
102591,2026-05-11 05:51:00,180.023,0.948
104031,2026-05-12 05:51:00,176.763,0.953
105471,2026-05-13 05:51:00,176.813,0.949
106911,2026-05-14 05:51:00,176.930,0.949
111231,2026-05-17 05:51:00,176.504,0.964
112671,2026-05-18 05:51:00,177.356,0.927
114111,2026-05-19 05:51:00,176.942,0.987
115551,2026-05-20 05:51:00,176.613,0.936
116991,2026-05-21 05:51:00,177.059,0.971
118431,2026-05-22 05:51:00,176.878,0.943


df_raw  : (129591, 45)
df_on   : (65610, 48)
df_diff : (65609, 48)


In [5]:
# S2 실행: 이 셀만 다시 돌리면 됩니다.
output_s2 = resolve_output_path(OUTPUT_S2_NAME)
s2_rows = []
for target, predictors in TARGET_DICTIONARY.items():
    level_row = run_ols_s2(df_on, target, predictors, tag="LEVEL_on")
    diff_row = run_ols_s2(df_diff, target, predictors, tag="DIFF_on")
    if level_row is not None:
        s2_rows.append(level_row)
    if diff_row is not None:
        s2_rows.append(diff_row)

s2_results = pd.DataFrame(s2_rows)
s2_results.to_csv(output_s2, index=False)

print(f"S2 저장 경로: {output_s2.resolve()}  rows={len(s2_results)}")
display(s2_results[["target", "variant", "N", "R2", "AdjR2", "DW", "max_vif", "quality_flags"]])


S2 저장 경로: C:\Users\ui203\OneDrive\문서\green_project\human_A\ols_first_pass_v2.csv  rows=10


,target,variant,N,R2,AdjR2,DW,max_vif,quality_flags
0,motor_current_a,LEVEL_on,65610,0.996421,0.996421,0.020914,142.856075,잔차 자기상관 심함 | 다중공선성 매우 큼 | 추세 또는 상태전이 주도 적합 의심
1,motor_current_a,DIFF_on,65609,0.991013,0.991013,0.311002,62.783784,잔차 자기상관 심함 | 다중공선성 매우 큼 | 추세 또는 상태전이 주도 적합 의심
2,zone1_resistance,LEVEL_on,65610,0.776974,0.776967,0.034997,3.674825,잔차 자기상관 심함
3,zone1_resistance,DIFF_on,65609,0.039883,0.039854,1.924981,2.204810,특이 경고 없음
4,wire_to_water_efficiency,LEVEL_on,65610,0.926198,0.926194,0.013622,6546.328518,잔차 자기상관 심함 | 다중공선성 매우 큼
5,wire_to_water_efficiency,DIFF_on,65609,0.898251,0.898245,0.256537,28.714228,잔차 자기상관 심함 | 다중공선성 큼
6,mix_ph,LEVEL_on,65610,0.000109,0.000048,0.040391,151.671932,잔차 자기상관 심함 | 다중공선성 매우 큼
7,mix_ph,DIFF_on,65609,0.000251,0.000190,0.917184,3.872192,잔차 자기상관 심함
8,mix_ec_ds_m,LEVEL_on,65610,0.763160,0.763138,0.037690,90237.398618,잔차 자기상관 심함 | 다중공선성 매우 큼
9,mix_ec_ds_m,DIFF_on,65609,0.003904,0.003813,0.666477,5.864764,잔차 자기상관 심함


In [6]:
# S1과 S2를 나란히 비교해 어떤 결과를 우선 해석할지 추천합니다.
comparison_rows = []
for target in TARGET_DICTIONARY:
    s1_row = s1_results.loc[s1_results["target"] == target].iloc[0].to_dict()
    level_row = s2_results.loc[(s2_results["target"] == target) & (s2_results["variant"] == "LEVEL_on")].iloc[0].to_dict()
    diff_row = s2_results.loc[(s2_results["target"] == target) & (s2_results["variant"] == "DIFF_on")].iloc[0].to_dict()
    recommended_variant, recommendation_reason = recommend_variant(level_row, diff_row)

    comparison_rows.append({
        "target": target,
        "S1_DW": s1_row["DW"],
        "LEVEL_on_DW": level_row["DW"],
        "DIFF_on_DW": diff_row["DW"],
        "S1_max_vif": s1_row["max_vif"],
        "LEVEL_on_max_vif": level_row["max_vif"],
        "DIFF_on_max_vif": diff_row["max_vif"],
        "권장 해석 버전": recommended_variant,
        "권장 이유": recommendation_reason,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


,target,S1_DW,LEVEL_on_DW,DIFF_on_DW,S1_max_vif,LEVEL_on_max_vif,DIFF_on_max_vif,권장 해석 버전,권장 이유
0,motor_current_a,0.018046,0.020914,0.311002,75.157341,142.856075,62.783784,추가 재설계 필요,차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다. 다만 자기상관 또는 공선성...
1,zone1_resistance,0.339391,0.034997,1.924981,3.673862,3.674825,2.204810,DIFF_on,차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다. 현재 진단상 가장 해석 가...
2,wire_to_water_efficiency,0.009025,0.013622,0.256537,9217.408279,6546.328518,28.714228,추가 재설계 필요,차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다. 다만 자기상관 또는 공선성...
3,mix_ph,0.035764,0.040391,0.917184,36.203439,151.671932,3.872192,DIFF_on (주의),차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다. 다만 잔차 자기상관이나 공...
4,mix_ec_ds_m,0.032915,0.037690,0.666477,75716.404243,90237.398618,5.864764,추가 재설계 필요,차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다. 다만 자기상관 또는 공선성...


## S3. `motor_current_a` / `wire_to_water_efficiency` VIF 정리

S3는 `S2`의 `DIFF_on` 결과를 다시 좁혀 보는 단계입니다.

여기서의 판단은 단순합니다.

1. `LEVEL_on`은 시간 추세 오염이 여전히 커서 더 이상 주력 비교 대상으로 쓰지 않습니다.
2. `DIFF_on`에서도 `motor_current_a`, `wire_to_water_efficiency`는 쌍공선 변수쌍이 남아 있습니다.
3. 그래서 한 번에 한 변수씩 제거한 후보 모델 두 개씩을 만들고, `v2_base`와 `AIC/BIC`, `DW`, `max_vif`를 같이 비교합니다.
   - 이름 읽는 법: `motor_current_a__v3a_drop_discharge` = `motor_current_a` 타깃에서 `v3a` 후보이며 `discharge_pressure_kpa`를 뺀 모델
4. 핵심은 **VIF가 낮아졌다는 이유만으로 채택하지 않는다**는 점입니다. 설명력 손실이 너무 크면 제거는 정당화되지 않습니다.
5. 현재 데이터에서는 두 타깃 모두 축소 모델이 `AIC`를 크게 악화시켜, 최종 판정은 `제거 비권장`입니다.


In [7]:
# S3 대상: DIFF_on에서도 공선성이 큰 두 타깃만 다시 정리합니다.
# 이름 읽는 법
# - <target>__v2_base            : S2의 DIFF_on 기본 모델
# - <target>__v3a_drop_xxx      : S3의 첫 번째 축소 후보, xxx 변수를 제거한 모델
# - <target>__v3b_drop_xxx      : S3의 두 번째 축소 후보, xxx 변수를 제거한 모델
# 예) motor_current_a__v3b_drop_rpm
#   -> motor_current_a 타깃
#   -> S3의 두 번째 후보(v3b)
#   -> pump_rpm을 제거한 모델
TARGET_DICTIONARY_V3 = {
    # motor_current_a
    # v2 base: motor_temperature_c, pump_rpm, discharge_pressure_kpa
    # DIFF_on VIF: motor_temp≈1.02, pump_rpm≈62.7, discharge_pressure≈62.6
    "motor_current_a__v3a_drop_discharge": [
        "motor_temperature_c",
        "pump_rpm",
    ],
    "motor_current_a__v3b_drop_rpm": [
        "motor_temperature_c",
        "discharge_pressure_kpa",
    ],

    # wire_to_water_efficiency
    # v2 base: pump_rpm, motor_temperature_c, suction_pressure_kpa, bearing_temperature_c
    # DIFF_on VIF: pump_rpm≈28.6, motor_temp≈1.24, suction_pressure≈28.5, bearing_temp≈1.24
    "wire_to_water_efficiency__v3a_drop_suction": [
        "pump_rpm",
        "motor_temperature_c",
        "bearing_temperature_c",
    ],
    "wire_to_water_efficiency__v3b_drop_rpm": [
        "motor_temperature_c",
        "suction_pressure_kpa",
        "bearing_temperature_c",
    ],
}


VARIANT_LABELS_S3 = {
    "motor_current_a__v2_base": "motor_current_a 기본모델(S2 DIFF_on)",
    "motor_current_a__v3a_drop_discharge": "motor_current_a 후보 A: discharge_pressure_kpa 제거",
    "motor_current_a__v3b_drop_rpm": "motor_current_a 후보 B: pump_rpm 제거",
    "wire_to_water_efficiency__v2_base": "wire_to_water_efficiency 기본모델(S2 DIFF_on)",
    "wire_to_water_efficiency__v3a_drop_suction": "wire_to_water_efficiency 후보 A: suction_pressure_kpa 제거",
    "wire_to_water_efficiency__v3b_drop_rpm": "wire_to_water_efficiency 후보 B: pump_rpm 제거",
}


def base_target(key):
    """후보 모델 이름에서 실제 타깃명을 분리합니다."""
    return key.split("__")[0]


def describe_s3_variant(key):
    """variant 문자열을 한국어 설명으로 바꿉니다."""
    return VARIANT_LABELS_S3.get(key, key)


# 필요한 중간 결과가 메모리에 없으면 S2 산출물을 다시 불러옵니다.
if "df_diff" not in globals():
    input_path_s3 = resolve_existing_path(INPUT_PATHS)
    df_raw_s3 = load_source_frame(input_path_s3)
    on_mask_s3, _, _ = diagnose_pump_on_rules(df_raw_s3)
    df_diff = add_derived_columns(df_raw_s3.loc[on_mask_s3].copy()).set_index("timestamp").diff().dropna()

if "s2_results" not in globals():
    s2_results = pd.read_csv(resolve_existing_output_path(OUTPUT_S2_NAME))


# S3는 S2의 DIFF_on 데이터와 HAC OLS 러너를 그대로 재사용합니다.
rows = []
for key, predictors in TARGET_DICTIONARY_V3.items():
    target = base_target(key)
    result = run_ols(df_diff, target, predictors, tag=key)
    if result is not None:
        rows.append(result)

s3_results = pd.DataFrame(rows)
s3_results["variant_설명"] = s3_results["variant"].map(describe_s3_variant)

# 비교 기준은 S2의 DIFF_on 기본 모델입니다.
base_s3 = s2_results[
    (s2_results["variant"] == "DIFF_on")
    & (s2_results["target"].isin(["motor_current_a", "wire_to_water_efficiency"]))
].copy()
base_s3["variant"] = base_s3["target"] + "__v2_base"

comparison_rows_s3 = []
for target in ["motor_current_a", "wire_to_water_efficiency"]:
    base_row = base_s3.loc[base_s3["target"] == target].iloc[0].to_dict()
    comparison_rows_s3.append({
        "variant": base_row["variant"],
        "variant_설명": describe_s3_variant(base_row["variant"]),
        "target": target,
        "N": base_row["N"],
        "R2": base_row["R2"],
        "AdjR2": base_row["AdjR2"],
        "DW": base_row["DW"],
        "AIC": base_row["AIC"],
        "BIC": base_row["BIC"],
        "max_vif": base_row["max_vif"],
        "delta_AIC_vs_base": 0.0,
        "delta_BIC_vs_base": 0.0,
        "판정": "기준 모델",
        "판정 이유": "비교 기준이 되는 S2 DIFF_on 기본 모델입니다.",
    })

    for _, candidate in s3_results.loc[s3_results["target"] == target].iterrows():
        delta_aic, delta_bic, verdict, reason = judge_s3_candidate(base_row, candidate)
        comparison_rows_s3.append({
            "variant": candidate["variant"],
            "variant_설명": describe_s3_variant(candidate["variant"]),
            "target": target,
            "N": candidate["N"],
            "R2": candidate["R2"],
            "AdjR2": candidate["AdjR2"],
            "DW": candidate["DW"],
            "AIC": candidate["AIC"],
            "BIC": candidate["BIC"],
            "max_vif": candidate["max_vif"],
            "delta_AIC_vs_base": delta_aic,
            "delta_BIC_vs_base": delta_bic,
            "판정": verdict,
            "판정 이유": reason,
        })

output_s3 = resolve_output_path(OUTPUT_S3_NAME)
s3_comparison = pd.DataFrame(comparison_rows_s3).sort_values(["target", "AIC"]).reset_index(drop=True)
s3_comparison.to_csv(output_s3, index=False)

print("\n===== S3 비교표 (AIC 오름차순 = 상대적으로 유리) =====")
print(s3_comparison.to_string(index=False))
print(f"\n저장: {output_s3.resolve()}")
display(s3_results[["variant", "variant_설명", "target", "N", "R2", "AdjR2", "DW", "AIC", "BIC", "max_vif", "pvals", "vif"]])



===== S3 비교표 (AIC 오름차순 = 상대적으로 유리) =====
                                   variant                                             variant_설명                   target     N       R2    AdjR2       DW            AIC            BIC   max_vif  delta_AIC_vs_base  delta_BIC_vs_base     판정                                 판정 이유
                  motor_current_a__v2_base                       motor_current_a 기본모델(S2 DIFF_on)          motor_current_a 65609 0.991013 0.991013 0.311002 -399965.295101 -399928.929228 62.783784           0.000000           0.000000  기준 모델        비교 기준이 되는 S2 DIFF_on 기본 모델입니다.
             motor_current_a__v3b_drop_rpm                      motor_current_a 후보 B: pump_rpm 제거          motor_current_a 65609 0.989703 0.989703 0.271734 -391037.074649 -391009.800244  1.023905        8928.220452        8919.128984 제거 비권장 VIF는 정리됐지만 AIC 손실이 커서 설명력 희생이 너무 큽니다.
       motor_current_a__v3a_drop_discharge        motor_current_a 후보 A: discharge_pressure_kpa 제거          motor_current_

,variant,variant_설명,target,N,R2,AdjR2,DW,AIC,BIC,max_vif,pvals,vif
0,motor_current_a__v3a_drop_discharge,motor_current_a 후보 A: discharge_pressure_kpa 제거,motor_current_a,65609,0.964917,0.964916,0.216534,-310610.946354,-310583.671949,1.023655,"{'motor_temperature_c': 0.3535103448637521, 'p...","{'motor_temperature_c': 1.0236550233619872, 'p..."
1,motor_current_a__v3b_drop_rpm,motor_current_a 후보 B: pump_rpm 제거,motor_current_a,65609,0.989703,0.989703,0.271734,-391037.074649,-391009.800244,1.023905,"{'motor_temperature_c': 0.0011423934287754056,...","{'motor_temperature_c': 1.02390454706557, 'dis..."
2,wire_to_water_efficiency__v3a_drop_suction,wire_to_water_efficiency 후보 A: suction_pressur...,wire_to_water_efficiency,65609,0.807816,0.807807,0.192277,-938877.113068,-938840.747196,1.253883,"{'pump_rpm': 1.4923954865105348e-251, 'motor_t...","{'pump_rpm': 1.0295870031758307, 'motor_temper..."
3,wire_to_water_efficiency__v3b_drop_rpm,wire_to_water_efficiency 후보 B: pump_rpm 제거,wire_to_water_efficiency,65609,0.685843,0.685828,0.193796,-906634.250329,-906597.884457,1.254427,{'motor_temperature_c': 2.8454999174550293e-09...,"{'motor_temperature_c': 1.2544274533398942, 's..."


## S5. lag feature 전진선택으로 자기상관 다시 점검

S5는 `df_diff` 위에 설명변수 lag를 쌓아서 `DW`가 개선되는지 확인하는 단계입니다.

여기서는 `zone1_resistance`를 제외한 4개 타깃만 다시 봅니다.

1. 각 설명변수의 `lag1`, `lag3`, `lag5`, `lag10` 컬럼을 생성합니다.
2. `lag 0` 현재 변수만 쓴 base 모델에서 시작합니다.
3. 아직 넣지 않은 lag 중 `AIC`를 가장 많이 낮추는 변수 1개만 한 번에 추가합니다.
4. `ΔAIC < -2`일 때만 채택하고, 더 이상 충분히 개선되지 않으면 멈춥니다.
5. 최종 모델의 `DW`, `VIF`, `p값`, `coef`를 저장하고 `S2 DIFF_on` base와 비교합니다.

샘플링 간격이 1분이므로 `lag1`, `lag3`, `lag5`, `lag10`은 각각 1분, 3분, 5분, 10분 시차를 뜻합니다.

결과 파일은 `ols_s5_lag_results.csv`, `ols_s5_lag_history.csv` 두 개로 저장합니다.


In [8]:
import numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# S5 셀만 다시 실행해도 되도록 필요한 입력을 먼저 확인합니다.
if "df_diff" not in globals():
    input_path_s5 = resolve_existing_path(INPUT_PATHS)
    df_raw_s5 = load_source_frame(input_path_s5)
    on_mask_s5, _, _ = diagnose_pump_on_rules(df_raw_s5)
    df_diff = add_derived_columns(df_raw_s5.loc[on_mask_s5].copy()).set_index("timestamp").diff().dropna()

# ---- 설정 ----
LAG_MINUTES = [1, 3, 5, 10]  # 테스트할 lag (분 단위 = 행 단위, 1분 샘플링)

# zone1_resistance는 이미 S2에서 통과했으므로 제외합니다.
TARGET_DICTIONARY_S5 = {
    "motor_current_a": [
        "motor_temperature_c",
        "pump_rpm",
        "discharge_pressure_kpa",
    ],
    "wire_to_water_efficiency": [
        "pump_rpm",
        "motor_temperature_c",
        "suction_pressure_kpa",
        "bearing_temperature_c",
    ],
    "mix_ph": [
        "dosing_acid_ml_min",
        "mix_temp_c",
        "acid_tank_level_pct",
        "mix_flow_l_min",
    ],
    "mix_ec_ds_m": [
        "mix_target_ec_ds_m",
        "tank_a_level_pct",
        "tank_b_level_pct",
        "mix_temp_c",
        "mix_flow_l_min",
        "raw_water_temp_c",
    ],
}


def add_lag_features(df, cols, lags):
    """원본 df에 cols × lags 조합의 lag 컬럼을 추가한다."""
    out = df.copy()
    for col in cols:
        for lag in lags:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)
    return out.dropna()


def vif_table(X):
    if X.shape[1] < 2:
        return {c: 1.0 for c in X.columns}
    v = X.astype(float).values
    return {c: float(variance_inflation_factor(v, i)) for i, c in enumerate(X.columns)}


def fit_ols(df, y_col, x_cols, tag=""):
    d = df[[y_col] + x_cols].dropna()
    if len(d) < 100:
        print(f"[{tag}] SKIP: N={len(d)}")
        return None
    X = sm.add_constant(d[x_cols], has_constant="add")
    res = sm.OLS(d[y_col], X).fit(cov_type="HAC", cov_kwds={"maxlags": 20})
    dw = sm.stats.stattools.durbin_watson(res.resid)
    jb_p = sm.stats.stattools.jarque_bera(res.resid)[1]
    vif = vif_table(d[x_cols])
    return {
        "tag": tag,
        "target": y_col,
        "xs": x_cols,
        "N": int(res.nobs),
        "R2": res.rsquared,
        "AdjR2": res.rsquared_adj,
        "DW": dw,
        "JB_p": jb_p,
        "AIC": res.aic,
        "BIC": res.bic,
        "max_vif": max(vif.values()),
        "pvals": {c: res.pvalues[c] for c in x_cols},
        "vif": vif,
        "coef": {c: res.params[c] for c in x_cols},
        "res": res,
    }


def forward_select_lags(df, y_col, base_xs, lags):
    """
    base_xs(lag 0)에서 시작.
    각 라운드에서 아직 안 넣은 lag 컬럼 중 AIC를 가장 많이 줄이는 것을 하나씩 추가합니다.
    AIC가 더 이상 충분히 감소하지 않으면 정지합니다.
    """
    lag_cols = [f"{col}_lag{lag}" for col in base_xs for lag in lags]
    available = [c for c in lag_cols if c in df.columns]

    current_xs = list(base_xs)
    best = fit_ols(df, y_col, current_xs, tag="base(lag0)")
    if best is None:
        return None, []
    best_aic = best["AIC"]
    history = [{
        "step": 0,
        "added": "—",
        "xs": list(current_xs),
        "AIC": best_aic,
        "DW": best["DW"],
        "AdjR2": best["AdjR2"],
    }]

    step = 0
    while available:
        step += 1
        candidates = []
        for cand in available:
            trial_xs = current_xs + [cand]
            r = fit_ols(df, y_col, trial_xs, tag=f"trial_{cand}")
            if r:
                candidates.append((cand, r))
        if not candidates:
            break

        candidates.sort(key=lambda x: x[1]["AIC"])
        best_cand, best_r = candidates[0]
        if best_r["AIC"] >= best_aic - 2:
            break

        current_xs.append(best_cand)
        available.remove(best_cand)
        best_aic = best_r["AIC"]
        best = best_r
        history.append({
            "step": step,
            "added": best_cand,
            "xs": list(current_xs),
            "AIC": best_aic,
            "DW": best["DW"],
            "AdjR2": best["AdjR2"],
        })
        print(
            f"  [{y_col}] step {step}: +{best_cand}  AIC={best_aic:.1f}  "
            f"DW={best['DW']:.3f}  AdjR²={best['AdjR2']:.4f}"
        )

    return best, history


# ---- 메인 실행 ----
# df_diff는 이전 셀에서 이미 생성됨 (펌프 on + 1차 차분)
all_base_cols = set()
for xs in TARGET_DICTIONARY_S5.values():
    all_base_cols.update(xs)

df_lag = add_lag_features(df_diff, list(all_base_cols), LAG_MINUTES)
print(f"df_lag shape: {df_lag.shape}  (lag {max(LAG_MINUTES)}행 소실)")

results = []
histories = {}
for y, base_xs in TARGET_DICTIONARY_S5.items():
    print(f"\n{'='*60}")
    print(f"S5 전진선택: {y}")
    print(f"{'='*60}")
    best, hist = forward_select_lags(df_lag, y, base_xs, LAG_MINUTES)
    if best:
        best.pop("res", None)
        results.append(best)
        histories[y] = hist

df_result = pd.DataFrame(results)
out1 = resolve_output_path(OUTPUT_S5_RESULTS_NAME)
df_result.to_csv(out1, index=False)
print(f"\n최종 모델 저장: {out1.resolve()}")

hist_rows = []
for y, hist in histories.items():
    for h in hist:
        hist_rows.append({"target": y, **h})

df_hist = pd.DataFrame(hist_rows)
out2 = resolve_output_path(OUTPUT_S5_HISTORY_NAME)
df_hist.to_csv(out2, index=False)
print(f"전진선택 과정 저장: {out2.resolve()}")

print("\n===== S5 최종 비교 (vs S2 DIFF_on base) =====")
v2 = pd.read_csv(resolve_output_path(OUTPUT_S2_NAME))
base = v2[v2["variant"] == "DIFF_on"][["target", "R2", "AdjR2", "DW", "AIC", "BIC"]].copy()
base.columns = ["target", "base_R2", "base_AdjR2", "base_DW", "base_AIC", "base_BIC"]
cmp = df_result[["target", "R2", "AdjR2", "DW", "AIC", "BIC", "max_vif", "xs", "pvals", "vif", "coef"]].merge(base, on="target", how="left")
cmp["ΔAIC"] = cmp["AIC"] - cmp["base_AIC"]
cmp["ΔDW"] = cmp["DW"] - cmp["base_DW"]
print(cmp[["target", "base_DW", "DW", "ΔDW", "base_AIC", "AIC", "ΔAIC", "AdjR2", "max_vif"]].to_string(index=False))
display(cmp)


df_lag shape: (65599, 100)  (lag 10행 소실)

S5 전진선택: motor_current_a
  [motor_current_a] step 1: +pump_rpm_lag10  AIC=-409951.3  DW=0.362  AdjR²=0.9923
  [motor_current_a] step 2: +discharge_pressure_kpa_lag10  AIC=-418853.6  DW=0.409  AdjR²=0.9932
  [motor_current_a] step 3: +pump_rpm_lag1  AIC=-419777.2  DW=0.402  AdjR²=0.9933
  [motor_current_a] step 4: +discharge_pressure_kpa_lag1  AIC=-420204.5  DW=0.389  AdjR²=0.9934
  [motor_current_a] step 5: +motor_temperature_c_lag1  AIC=-420557.9  DW=0.378  AdjR²=0.9934
  [motor_current_a] step 6: +motor_temperature_c_lag10  AIC=-420758.2  DW=0.372  AdjR²=0.9934
  [motor_current_a] step 7: +motor_temperature_c_lag5  AIC=-420763.7  DW=0.372  AdjR²=0.9934

S5 전진선택: wire_to_water_efficiency
  [wire_to_water_efficiency] step 1: +pump_rpm_lag10  AIC=-986503.7  DW=0.275  AdjR²=0.9066
  [wire_to_water_efficiency] step 2: +suction_pressure_kpa_lag10  AIC=-992670.2  DW=0.319  AdjR²=0.9150
  [wire_to_water_efficiency] step 3: +pump_rpm_lag1  AIC=-993007

,target,R2,AdjR2,DW,AIC,BIC,max_vif,xs,pvals,vif,coef,base_R2,base_AdjR2,base_DW,base_AIC,base_BIC,ΔAIC,ΔDW
0,motor_current_a,0.993440,0.993439,0.372472,-420763.743046,-420663.738573,316.317945,"[motor_temperature_c, pump_rpm, discharge_pres...",{'motor_temperature_c': 1.0726513231698263e-06...,"{'motor_temperature_c': 1.3815866093927576, 'p...","{'motor_temperature_c': -0.02058121470281486, ...",0.991013,0.991013,0.311002,-399965.295101,-399928.929228,-20798.447946,0.061470
1,wire_to_water_efficiency,0.916465,0.916445,0.305164,-993816.470245,-993661.917878,135.058271,"[pump_rpm, motor_temperature_c, suction_pressu...","{'pump_rpm': 5.037182250414346e-219, 'motor_te...","{'pump_rpm': 135.0582714620089, 'motor_tempera...","{'pump_rpm': 3.567727473915135e-05, 'motor_tem...",0.898251,0.898245,0.256537,-980598.934419,-980553.477078,-13217.535827,0.048627
2,mix_ph,0.001062,0.000940,0.916720,-721319.419344,-721237.597502,6.523788,"[dosing_acid_ml_min, mix_temp_c, acid_tank_lev...","{'dosing_acid_ml_min': 0.5431634641223105, 'mi...","{'dosing_acid_ml_min': 6.523788256454477, 'mix...","{'dosing_acid_ml_min': 0.0001039938856843784, ...",0.000251,0.000190,0.917184,-721391.104325,-721345.646984,71.684981,-0.000463
3,mix_ec_ds_m,0.006394,0.006152,0.667549,-664294.738814,-664140.186447,7.572396,"[mix_target_ec_ds_m, tank_a_level_pct, tank_b_...","{'mix_target_ec_ds_m': 1.5122904075276214e-05,...","{'mix_target_ec_ds_m': 1.1704820381313354, 'ta...","{'mix_target_ec_ds_m': 0.10268862967634915, 't...",0.003904,0.003813,0.666477,-664252.448714,-664188.808437,-42.290101,0.001072
